Import libraries and modules

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

Load dataset and examine structure

In [ ]:
url = "https://raw.githubusercontent.com/UmarHMoh/COMP3608-LOANS-PROJECT/refs/heads/datasets-Load_Default.csv/data/Loan_Default.csv"

#df=pd.read_csv("Loan_Default.csv")
df = pd.read_csv(url, encoding="latin1", sep=",")
print(df.shape)
print(df.head())

We collect pre-cleaning data and export to Excel which will help to inform decision making as it relates to cleaning the dataset.

In [ ]:
summary=pd.DataFrame()
# All variables
summary["Variable Name"]=df.columns
summary["Data Type"]=df.dtypes.values
summary["No. Unique Values"] = df.nunique().values
summary["Sample Values"] = summary["Variable Name"].map(
    lambda col: ", ".join(map(str, df[col].dropna().unique()[:5]))
)
summary["Missing %"] = summary["Variable Name"].map(df.isnull().mean() * 100)
# Categorical columns
cat_cols = df.select_dtypes(include=['object', 'string']).columns
summary["Top Category Frequency (%)"] = None
for col in cat_cols:
    top_freq = df[col].value_counts(normalize=True).iloc[0] * 100
    summary.loc[summary["Variable Name"] == col, "Top Category Frequency (%)"] = top_freq
#Numerical Columns
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
summary["Num: Std"] = summary["Variable Name"].map(df.std(numeric_only=True))
summary["Num: Min"] = summary["Variable Name"].map(df.min(numeric_only=True))
summary["Num: Max"] = summary["Variable Name"].map(df.max(numeric_only=True))
correlations = df.corr(numeric_only=True)["Status"]
summary["Correlation with Status"] = summary["Variable Name"].map(correlations)
#print(summary)
summary.to_excel("data_summary.xlsx", index=False)

Drop columns with low variability; Define target y and features matrix X; For selected variables with missing values, we add missing indicators column to X

In [ ]:
df=df.drop(columns=["Security_Type","total_units","Secured_by","construction_type","year","ID"])
y = df["Status"]
X = df.drop("Status", axis=1)
X["interest_missing"] = X["rate_of_interest"].isnull().astype(int)
X["property_value_missing"] = X["property_value"].isnull().astype(int)
X["income_missing"] = X["income"].isnull().astype(int)
X["dtir1_missing"] = X["dtir1"].isnull().astype(int)

We split X and y into train and test sets. We define numerical columns and categorical columns of X. 

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object"]).columns

Perform imputation for missing values

In [ ]:
# Numeric
num_imputer = SimpleImputer(strategy="median")
num_imputer.fit(X_train[num_cols])
X_train[num_cols] = num_imputer.transform(X_train[num_cols])
X_test[num_cols] = num_imputer.transform(X_test[num_cols])
# Categorical
cat_imputer = SimpleImputer(strategy="most_frequent")
cat_imputer.fit(X_train[cat_cols])
X_train[cat_cols] = cat_imputer.transform(X_train[cat_cols])
X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])

Encoding of categorical variables. (For decision trees-encoding is optional, for logistic regression and KNN, encoding is very critical)

In [ ]:
#For encoding of categorical variables, we split into nominal and ordinal
cat_ord_cols = ["age"]
cat_nom_cols = [col for col in cat_cols if col not in cat_ord_cols]

#One Hot Encoding of nominal vars
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
#1. Create arrays captured encoded columns
X_train_nom_encoded = ohe.fit_transform(X_train[cat_nom_cols])
X_test_nom_encoded = ohe.transform(X_test[cat_nom_cols])
#2. Convert arrays to dataframes
train_nom_df = pd.DataFrame(X_train_nom_encoded, columns=ohe.get_feature_names_out(cat_nom_cols), index=X_train.index)
test_nom_df = pd.DataFrame(X_test_nom_encoded, columns=ohe.get_feature_names_out(cat_nom_cols), index=X_test.index)
#3. Drop original nominal columns from the DFs
X_train = X_train.drop(columns=cat_nom_cols)
X_test = X_test.drop(columns=cat_nom_cols)
#4. Add in the encoded columns
X_train = pd.concat([X_train, train_nom_df], axis=1)
X_test = pd.concat([X_test, test_nom_df], axis=1)
#will now have encoded nominal variables

#Ordinal Encoding
age_order = [['<25', '25-34', '35-44', '45-54', '55-64','65-74','>74']]
#the unique values of X_train["age"] after imputation were printed to populate this list
ord_encoder = OrdinalEncoder(categories=age_order)
X_train[cat_ord_cols] = ord_encoder.fit_transform(X_train[cat_ord_cols]) #reassigns to X_train ordinal columns
X_test[cat_ord_cols] = ord_encoder.transform(X_test[cat_ord_cols]) #reassigns to X_test ordinal columns

We now write code to scale the numeric variables. Many models rely on distances and gradients and having one feature that has possibly larger values will lead to its dominance in the model and erroneous modelling. (For decision trees-scaling is optional, for logistic regression and KNN, scaling is very critical). It is important that we exclude the binary missing indicators columns from num_cols as they should not be scaled. 

In [ ]:
scaler = StandardScaler()
#Note that num_cols excludes the OHE columns that were added but includes missing indicator columns which must be removed
indicator_cols = [
    "interest_missing",
    "property_value_missing",
    "income_missing",
    "dtir1_missing"
]
num_cols = [col for col in num_cols if col not in indicator_cols]
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])
print("Ran")

We now have X_train, X_test, y_train, y_test which are ready to be passed into our models.